# Lesson 6 — Variational Quantum Eigensolvers

**Quantum Computing with Qiskit II**

This notebook accompanies Lesson 6. Work through the cells in order. Every cell is meant to be run; modify the code freely and re-run to experiment.

**Topics covered:**
- The variational principle and upper bound on ground-state energy
- Measuring expectation values via Pauli decomposition
- Hardware-efficient ansatz design with `ParameterVector`
- The VQE optimization loop with COBYLA
- Convergence analysis and ansatz depth study
- `StatevectorEstimator` as a portable energy evaluation primitive

In [ ]:
!pip install qiskit qiskit-aer scipy pylatexenc --quiet

In [ ]:
import qiskit
import qiskit_aer

print("Qiskit:    ", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector

print("Imports complete.")

## 1. TFIM Hamiltonian (recap from Lesson 5)

The 4-qubit transverse-field Ising model from Lesson 5 is the running example:

$$H = J \sum_{i=0}^{n-2} Z_i Z_{i+1} + g \sum_{i=0}^{n-1} X_i$$

with $J = 1.0$ (ZZ coupling) and $g = 0.5$ (transverse field). The goal of VQE is to find the lowest eigenvalue of this Hamiltonian using a parameterized circuit.

In [ ]:
n = 4
J = 1.0
g = 0.5

pauli_terms, coeffs = [], []

# ZZ terms: Z_i Z_{i+1} for i = 0, 1, ..., n-2
for i in range(n - 1):
    s = ['I'] * n
    s[n - 1 - i]       = 'Z'
    s[n - 1 - (i + 1)] = 'Z'
    pauli_terms.append(''.join(s))
    coeffs.append(J)

# X terms: X_i for i = 0, 1, ..., n-1
for i in range(n):
    s = ['I'] * n
    s[n - 1 - i] = 'X'
    pauli_terms.append(''.join(s))
    coeffs.append(g)

H_op     = SparsePauliOp(pauli_terms, coeffs=coeffs)
H_matrix = H_op.to_matrix()

print("TFIM Hamiltonian (n=4, J=1.0, g=0.5):")
for term, coeff in zip(pauli_terms, coeffs):
    print(f"  {coeff:+.1f} * {term}")
# Expected: 7 terms total.
# ZZ terms: IIZZ, IZZI, ZZII  (coefficient +1.0)
# X  terms: IIIX, IIXI, IXII, XIII  (coefficient +0.5)

In [ ]:
# Exact ground state via full diagonalization
eigenvalues, eigenvectors = np.linalg.eigh(H_matrix)  # sorted real eigenvalues
E_exact = eigenvalues[0]
psi_gs  = eigenvectors[:, 0]  # ground-state eigenvector

print(f"Exact ground-state energy:  {E_exact:.6f}")
print(f"First excited-state energy: {eigenvalues[1]:.6f}")
print(f"Spectral width (max - min): {eigenvalues[-1] - eigenvalues[0]:.6f}")
# Expected: ground-state energy ≈ -3.427000
# The VQE target: find a parameterized state whose energy matches this value.

## 2. Hardware-Efficient Ansatz

The **hardware-efficient ansatz (HEA)** alternates parameterized $R_y$ rotation layers with fixed CX entangling layers. For $n$ qubits and $L$ layers the circuit has $n(L+1)$ parameters.

The function returns both the circuit and the `ParameterVector` so that parameters can be bound explicitly by index, avoiding issues with Qiskit's internal lexicographic parameter ordering.

In [ ]:
def hardware_efficient_ansatz(n, num_layers):
    # Hardware-efficient ansatz: Ry rotations interleaved with CX entanglers.
    # Returns: (circuit, parameter_vector)
    num_params = n * (num_layers + 1)
    theta = ParameterVector('theta', num_params)
    qc = QuantumCircuit(n)
    param_idx = 0
    for _ in range(num_layers):
        for i in range(n):             # rotation layer
            qc.ry(theta[param_idx], i)
            param_idx += 1
        for i in range(n - 1):        # entangling layer
            qc.cx(i, i + 1)
    for i in range(n):                # final rotation layer
        qc.ry(theta[param_idx], i)
        param_idx += 1
    return qc, theta


def bind_params(ansatz, theta, params):
    # Bind a numpy array to the ParameterVector theta in index order.
    # Using a dict keyed by theta[i] avoids Qiskit's internal lexicographic
    # parameter ordering, which can scramble values for parameter counts >= 10.
    return ansatz.assign_parameters({theta[i]: params[i]
                                     for i in range(len(params))})


# Build the L=2 ansatz used throughout this notebook
num_layers = 2
num_params = n * (num_layers + 1)   # 4 * 3 = 12
ansatz, theta = hardware_efficient_ansatz(n, num_layers)

print(f"Ansatz: n={n}, L={num_layers}, num_params={num_params}")
print(f"Gate counts: {dict(ansatz.count_ops())}")
print(f"Circuit depth: {ansatz.depth()}")
# Expected: num_params=12
# Gate counts: {'ry': 12, 'cx': 6}
# Depth: 8
#
# Why 8 and not 7: the entangling layer CX(0,1), CX(1,2), CX(2,3) cascades.
# CX(1,2) waits for qubit 1 freed by CX(0,1); CX(2,3) waits for qubit 2 freed
# by CX(1,2). So each entangling block has depth n-1 = 3, not 2.
# Formula for n=4: depth = (n-1)*L + (L+1) = 3L + (L+1) ... simplified:
# depth = 3L + 2  ->  L=1: 5,  L=2: 8,  L=3: 11,  L=4: 14.


In [ ]:
ansatz.draw('mpl')

## 3. Energy Evaluation

The variational energy $E(\theta) = \langle \psi(\theta) | H | \psi(\theta) \rangle$ is computed by:

1. Binding the parameters to produce a concrete (parameter-free) circuit.
2. Simulating the circuit to obtain the statevector $|\psi(\theta)\rangle$.
3. Calling `Statevector.expectation_value(H_op)` which evaluates the full sum $\sum_P c_P \langle P \rangle$ exactly in a single call.

In [ ]:
def energy(params):
    # Compute E(theta) = <psi(theta)|H|psi(theta)> via exact statevector simulation.
    bound = bind_params(ansatz, theta, params)
    sv    = Statevector(bound)
    return sv.expectation_value(H_op).real


# Sanity check at theta = 0:
# All Ry(0) = I, so the state is |0000>.
# <0000|Z_i Z_{i+1}|0000> = (+1)(+1) = +1 for each of the 3 ZZ bonds.
# <0000|X_i|0000>         = <0|X|0>  = 0 for each site.
# E(0) = J * 3 * 1 + g * 4 * 0 = 1.0 * 3 = 3.0
params_zero = np.zeros(num_params)
e_zero = energy(params_zero)
print(f"Energy at theta=0 (state |0000>):  {e_zero:.6f}")
print(f"  (Expected: 3.0 -- all ZZ bonds give +1, X terms vanish)")
print()

# Verify the variational bound: E(theta) >= E_exact for any theta
rng_check = np.random.default_rng(0)
violations = 0
for _ in range(20):
    e_rand = energy(rng_check.uniform(-np.pi, np.pi, num_params))
    if e_rand < E_exact - 1e-10:
        violations += 1
print(f"Variational bound checks: {20 - violations}/20 satisfied")
print(f"  (All 20 should pass -- the bound E(theta) >= E_exact always holds)")

## 4. VQE Optimization Loop

VQE minimizes $E(\theta)$ using a classical optimizer. COBYLA is a derivative-free method: it does not require gradient information, making it robust to small numerical noise and simple to configure.

The `history` list records the energy at each function evaluation so we can visualize convergence afterward.

In [ ]:
history = []

def cost(params):
    e = energy(params)
    history.append(e)
    return e


# Random initialization avoids the flat-gradient region at theta=0
rng = np.random.default_rng(42)
theta_init = rng.uniform(-np.pi, np.pi, num_params)
print(f"Initial energy: {energy(theta_init):.6f}")
print(f"Exact target:   {E_exact:.6f}")
print()
print("Running COBYLA optimizer ...")
result = minimize(cost, theta_init, method='COBYLA',
                  options={'maxiter': 5000, 'rhobeg': 0.5})
print("Done.")


In [ ]:
vqe_energy   = result.fun
theta_opt    = result.x
energy_error = abs(vqe_energy - E_exact)

print(f"VQE converged energy:       {vqe_energy:.6f}")
print(f"Exact ground-state energy:  {E_exact:.6f}")
print(f"Error |E_VQE - E0|:         {energy_error:.2e}")
print(f"Relative error:             {energy_error / abs(E_exact) * 100:.3f}%")
print(f"Function evaluations:       {result.nfev}")
# Expected: VQE energy close to -3.427000, relative error < 0.1% for L=2.
# Function evaluations: typically 500 to 2000 with COBYLA and 12 parameters.
# If relative error is above 1%, re-run this cell: COBYLA can get stuck in a
# local minimum depending on the random starting point.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history, color='steelblue', linewidth=1.0, alpha=0.8,
        label='$E(\\theta)$ per evaluation')
ax.axhline(E_exact, color='tomato', linewidth=1.5, linestyle='--',
           label=f'Exact $E_0 = {E_exact:.4f}$')
ax.set_xlabel("Function evaluation", fontsize=12)
ax.set_ylabel("Energy", fontsize=12)
ax.set_title(
    f'VQE convergence (TFIM $n=4$, $J=1$, $g=0.5$, $L={num_layers}$)',
    fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(E_exact - 0.5, max(history[:20]) + 0.5)
plt.tight_layout()
plt.show()
# Expected: energy starts near the random initial value and decreases
# (with some exploration noise from COBYLA), converging toward the
# dashed red line at E_exact ≈ -3.4270.

## 5. Verifying the Optimized State

The energy error tells us how close the variational energy is to the exact value. The **infidelity** $1 - |\langle \psi_\mathrm{exact} | \psi_\mathrm{VQE} \rangle|^2$ measures how close the actual statevector is to the exact ground-state eigenvector.

In [ ]:
psi_vqe = Statevector(bind_params(ansatz, theta, theta_opt)).data

overlap    = abs(np.dot(psi_gs.conj(), psi_vqe))**2
infidelity = 1.0 - overlap

print(f"Overlap |<psi_exact|psi_VQE>|^2:  {overlap:.6f}")
print(f"Infidelity:                        {infidelity:.2e}")
print()
print(f"Energy from psi_VQE:               {energy(theta_opt):.6f}")
print(f"Exact E0:                          {E_exact:.6f}")
# Expected: overlap close to 1.0, infidelity < 1e-3 for L=2.
#
# Note: if the ground state is degenerate, the VQE state may converge to a
# different linear combination of degenerate eigenvectors. In that case the
# infidelity can be nonzero even when the energy matches exactly.
# Energy error is a more reliable convergence metric than infidelity.

## 6. Ansatz Depth Study

Increasing $L$ gives the ansatz more expressive power but also more parameters and greater circuit depth. For the 4-qubit TFIM we compare the converged VQE energy for $L = 1, 2, 3, 4$ and plot the energy error on a semilog axis.

In [ ]:
def make_cost(a, t, H):
    # Factory that returns a cost function closed over a specific ansatz.
    # Using a factory avoids Python's closure-by-reference gotcha in loops.
    def cost_fn(p):
        bound = bind_params(a, t, p)
        return Statevector(bound).expectation_value(H).real
    return cost_fn


layer_results = {}
rng_depth = np.random.default_rng(42)

print(f"{'L':>2}  {'params':>7}  {'depth':>6}  {'E_VQE':>12}  {'error':>10}  {'nfev':>6}")
print("-" * 54)

for L in [1, 2, 3, 4]:
    np_l           = n * (L + 1)
    ans_l, theta_l = hardware_efficient_ansatz(n, L)
    init_l         = rng_depth.uniform(-np.pi, np.pi, np_l)

    res_l = minimize(make_cost(ans_l, theta_l, H_op), init_l,
                     method='COBYLA',
                     options={'maxiter': 5000, 'rhobeg': 0.5})

    err = abs(res_l.fun - E_exact)
    layer_results[L] = {'energy': res_l.fun, 'error': err,
                        'nfev': res_l.nfev, 'depth': ans_l.depth(), 'params': np_l}
    print(f"  {L}  {np_l:7d}  {ans_l.depth():6d}  "
          f"{res_l.fun:12.6f}  {err:10.2e}  {res_l.nfev:6d}")

# Expected depths (n=4, formula 3L+2): L=1:5, L=2:8, L=3:11, L=4:14
# CX(0,1)->CX(1,2)->CX(2,3) cascade (each shares a qubit with the next),
# so the entangling block adds n-1=3 depth levels per layer, not 2.
#
# Expected energies/errors (approximate; vary with optimizer run):
# L=1:  8 params, depth= 5,  energy ~ -3.38 to -3.42,  error ~ 1e-2 to 5e-2
# L=2: 12 params, depth= 8,  energy ~ -3.42 to -3.427, error ~ 1e-3 to 1e-2
# L=3: 16 params, depth=11,  energy ~ -3.425 to -3.427, error ~ 1e-4 to 1e-3
# L=4: 20 params, depth=14,  energy ~ -3.4270,          error < 1e-3


In [ ]:
L_vals  = sorted(layer_results)
errors  = [layer_results[L]['error'] for L in L_vals]
one_pct = abs(E_exact) * 0.01   # 1% of |E0|

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(L_vals, errors, 'o-', color='steelblue', linewidth=2, markersize=8,
            label='VQE energy error')
ax.axhline(one_pct, color='gray', linestyle=':', linewidth=1.5,
           label=f'1% of $|E_0|$ = {one_pct:.3f}')
ax.set_xlabel("Number of ansatz layers $L$", fontsize=12)
ax.set_ylabel("Energy error $|E_\\mathrm{VQE} - E_0|$", fontsize=12)
ax.set_title('VQE convergence vs ansatz depth (TFIM $n=4$, $J=1$, $g=0.5$)', fontsize=12)
ax.set_xticks(L_vals)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
# Expected: error decreases by roughly an order of magnitude per added layer.
# The dotted line marks the 1% threshold; the minimum L that sits below it
# identifies the depth at which the ansatz becomes practically useful.

## 7. StatevectorEstimator Primitive

Qiskit's `StatevectorEstimator` provides the same energy evaluation through the Estimator interface used by Qiskit Runtime. The **PUB (Primitive Unified Bloc)** passed to `estimator.run` is a tuple `(circuit, observable)` for pre-bound circuits.

On Qiskit Runtime with real hardware, replacing `StatevectorEstimator` with the cloud `Estimator` from `qiskit_ibm_runtime` requires only changing the import and supplying a backend. The rest of the VQE code is identical.

In [ ]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()


def energy_estimator(params):
    # Evaluate energy using StatevectorEstimator.
    # Parameters are bound manually before passing to the Estimator so that
    # the circuit has no remaining free parameters (avoids the ParameterView
    # lexicographic ordering issue for parameter counts >= 10).
    bound = bind_params(ansatz, theta, params)  # pre-bind: no open parameters
    pub   = (bound, H_op)                       # PUB: (circuit, observable)
    job   = estimator.run([pub])
    return float(job.result()[0].data.evs)      # scalar expectation value


# Verify: Estimator matches direct Statevector evaluation at the optimal parameters
e_sv  = energy(theta_opt)
e_est = energy_estimator(theta_opt)
print(f"Statevector energy:  {e_sv:.10f}")
print(f"Estimator energy:    {e_est:.10f}")
print(f"Difference:          {abs(e_sv - e_est):.2e}")
# Expected: both values identical to floating-point precision (difference < 1e-14).

In [ ]:
# Run a short VQE using energy_estimator to confirm it works as a drop-in replacement
history_est = []

def cost_est(params):
    e = energy_estimator(params)
    history_est.append(e)
    return e

rng_est  = np.random.default_rng(7)
init_est = rng_est.uniform(-np.pi, np.pi, num_params)

result_est = minimize(cost_est, init_est, method='COBYLA',
                      options={'maxiter': 500, 'rhobeg': 0.5})

print(f"VQE (Estimator) converged energy: {result_est.fun:.6f}")
print(f"Exact ground-state energy:        {E_exact:.6f}")
print(f"Error:                            {abs(result_est.fun - E_exact):.2e}")
# Expected: converged energy close to -3.4270, same accuracy as the Statevector VQE.
# The two backends use identical math on the simulator.

## Exercises

Work through each exercise in the cell below it. Expected results are provided so you can check your work.

### Exercise 1: Ansatz Depth and the 1% Threshold

Using the `layer_results` dictionary computed in cell-19:

1. Compute the relative energy error $|E_\mathrm{VQE} - E_0| / |E_0|$ for each $L$.
2. Identify the smallest $L$ for which the relative error is less than 1%.
3. Report the total number of parameters and CX gate count for that circuit.

*Expected: the threshold is crossed at $L = 2$ or $L = 3$ depending on the optimizer run.*

In [ ]:
# Exercise 1: Minimum L for 1% relative error

# TODO: loop over layer_results and compute relative error
#
# for L, res in sorted(layer_results.items()):
#     rel_err = res['error'] / abs(E_exact)
#     print(f"L={L}: relative error = {rel_err:.3%}")

# TODO: identify the smallest L with relative error < 0.01
# Hint: build a fresh circuit with hardware_efficient_ansatz(n, L)
# and call .count_ops().get('cx', 0) to get the CX count

### Exercise 2: Optimizer Comparison

Run the $L=2$ VQE with `method='L-BFGS-B'` and `method='Nelder-Mead'` in addition to COBYLA (history already stored in `history`).

1. Run each optimizer from the same starting point `theta_init` (seed 42, already defined).
2. Track the energy history for each method.
3. Plot all three convergence histories on the same axes (energy vs function evaluation count).
4. Which method reaches $E < -3.4$ first?

*Expected: L-BFGS-B converges in fewer evaluations on the noiseless simulator. Nelder-Mead may need more evaluations but is also derivative-free.*

In [ ]:
# Exercise 2: Optimizer comparison

# TODO: run VQE with L-BFGS-B and Nelder-Mead from theta_init
#
# history_lbfgs, history_nm = [], []
#
# def cost_lbfgs(p):
#     e = energy(p); history_lbfgs.append(e); return e
#
# def cost_nm(p):
#     e = energy(p); history_nm.append(e); return e
#
# result_lbfgs = minimize(cost_lbfgs, theta_init, method='L-BFGS-B')
# result_nm    = minimize(cost_nm,    theta_init, method='Nelder-Mead')

# TODO: plot all three convergence histories
# Add a horizontal line at E = -3.4 to mark the target

### Exercise 3: Parameter Shift Gradients

Implement the parameter shift rule for the hardware-efficient ansatz:

$$\frac{\partial E}{\partial \theta_k} = \frac{1}{2}\left[E\!\left(\boldsymbol{\theta} + \frac{\pi}{2}\hat{\mathbf{e}}_k\right) - E\!\left(\boldsymbol{\theta} - \frac{\pi}{2}\hat{\mathbf{e}}_k\right)\right]$$

1. For `theta_init`, compute the full gradient vector using the parameter shift rule (requires $2p$ energy evaluations for $p = 12$ parameters).
2. Compute the gradient by finite differences: $[E(\theta + \epsilon e_k) - E(\theta - \epsilon e_k)] / (2\epsilon)$ with $\epsilon = 10^{-5}$.
3. Verify that the two gradients agree to within $10^{-4}$ in the $\ell_\infty$ norm.

*Expected: max absolute difference between the two gradient vectors $< 10^{-6}$ (parameter shift is exact; finite differences have $O(\epsilon^2)$ truncation error).*

In [ ]:
# Exercise 3: Parameter shift gradients

# TODO: implement parameter shift gradient
#
# def param_shift_gradient(params):
#     grad  = np.zeros(len(params))
#     shift = np.pi / 2
#     for k in range(len(params)):
#         e_k      = np.zeros(len(params)); e_k[k] = 1.0
#         grad[k]  = 0.5 * (energy(params + shift * e_k)
#                           - energy(params - shift * e_k))
#     return grad

# TODO: implement finite-difference gradient with eps=1e-5
#
# def fd_gradient(params, eps=1e-5):
#     grad = np.zeros(len(params))
#     for k in range(len(params)):
#         e_k = np.zeros(len(params)); e_k[k] = 1.0
#         grad[k] = (energy(params + eps * e_k) - energy(params - eps * e_k)) / (2 * eps)
#     return grad

# TODO: compare the two gradient vectors at theta_init
# grad_ps = param_shift_gradient(theta_init)
# grad_fd = fd_gradient(theta_init)
# print(f'Max absolute difference: {np.max(np.abs(grad_ps - grad_fd)):.2e}')

### Exercise 4: Excited-State Estimation via Deflation

After converging to the ground state $|\psi_0\rangle$, add a penalty term to the cost function:

$$C(\boldsymbol{\theta}) = E(\boldsymbol{\theta}) + \beta \, |\langle \psi_0 | \psi(\boldsymbol{\theta}) \rangle|^2$$

with $\beta = 10$. The penalty drives the optimizer away from the ground-state subspace, so the minimum of $C$ corresponds to the first excited state.

1. Run the deflation VQE from a fresh random starting point (seed 99).
2. Compare the converged energy to the exact first excited-state eigenvalue `eigenvalues[1]`.

*Expected: converged energy matches `eigenvalues[1]` to within the same accuracy as the ground-state run.*

In [ ]:
# Exercise 4: Excited-state estimation via deflation

# TODO: extract the ground-state VQE vector
# psi_gs_vqe = Statevector(bind_params(ansatz, theta, theta_opt)).data

# TODO: define the deflated cost function
#
# beta = 10.0
#
# def cost_deflated(params):
#     bound   = bind_params(ansatz, theta, params)
#     sv      = Statevector(bound).data
#     e       = (sv.conj() @ H_matrix @ sv).real
#     penalty = beta * abs(np.dot(psi_gs_vqe.conj(), sv))**2
#     return e + penalty

# TODO: run VQE with cost_deflated from a fresh starting point
# rng_ex4  = np.random.default_rng(99)
# init_ex4 = rng_ex4.uniform(-np.pi, np.pi, num_params)
# result_ex4 = minimize(cost_deflated, init_ex4, method='COBYLA',
#                        options={'maxiter': 1000, 'rhobeg': 0.5})
# print(f'Deflation VQE energy:         {result_ex4.fun:.6f}')
# print(f'Exact first excited-state:    {eigenvalues[1]:.6f}')
# print(f'Error:                        {abs(result_ex4.fun - eigenvalues[1]):.2e}')